# 201 · Self-describing vs schema-dependent

Companion to [Self-describing vs schema](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/self-describing-vs-schema-dependent/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/self_describing_vs_schema.ipynb)

**Goal:** measure how field names/type tags travel with JSON/MessagePack vs field numbers on a mini Protobuf-style encoding.

**Why this lab:** “binary vs text” is the wrong axis if you care about *where meaning lives*—in the payload or in a shared contract.

**How to use:** encode one record four ways (JSON, MessagePack map/array, field-number sketch); compare sizes and whether keys appear as UTF-8 on the wire.

**Expect:** JSON and MessagePack maps contain key strings; MessagePack array and Protobuf-style sketch do not; sizes drop as metadata leaves the message.

> **Honesty banner:** sizes and timings here are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth. Compare within one language and paradigm—not global format rankings.


In [ ]:
import json

RECORD = {"user_id": 42, "name": "Ada", "balance": 100}

try:
    import msgpack
    HAS_MSGPACK = True
except ImportError:
    HAS_MSGPACK = False
    print("pip install msgpack  # optional for MessagePack cells")



## JSON — names on every message

**Why:** JSON is the textbook self-describing case: structure and names ride with the data.

**How:** compact-encode a small record; assert each key appears as UTF-8 in the bytes.

**Expect:** readable object; keys visible in hex; larger than schema-dependent forms.

**Why it matters:** inspectability and loose coupling—at the cost of repeating metadata every message.


In [ ]:
def hex_bytes(b: bytes) -> str:
    return " ".join(f"{x:02x}" for x in b)

j = json.dumps(RECORD, separators=(",", ":")).encode("utf-8")
print(j.decode())
print("nbytes", len(j))
print("hex", hex_bytes(j)[:80], "…")
# Field names are UTF-8 substrings on the wire:
for key in RECORD:
    assert key.encode() in j
print("OK: each key appears as UTF-8 in the payload")



## MessagePack — type tags (+ keys if map)

**Why:** MessagePack is still a dynamic model—binary, not “schema-free of metadata.”

**How:** pack as a map (keys on wire) and as an array (order is the contract).

**Expect:** map still contains key strings; array is tiny and keyless—more schema-like discipline without an IDL file.

**Why it matters:** “we switched to binary” may still ship field names every time if you use maps.


In [ ]:
if not HAS_MSGPACK:
    print("SKIP msgpack")
else:
    packed = msgpack.packb(RECORD, use_bin_type=True)
    print("nbytes", len(packed))
    print("hex", hex_bytes(packed))
    # keys still present for map encoding
    for key in RECORD:
        assert key.encode() in packed
    print("OK: map keys still travel with MessagePack maps")
    # array form drops names — schema-like discipline without IDL
    as_array = msgpack.packb([RECORD["user_id"], RECORD["name"], RECORD["balance"]], use_bin_type=True)
    print("as array nbytes", len(as_array), "hex", hex_bytes(as_array))
    for key in RECORD:
        assert key.encode() not in as_array
    print("OK: array form has no field names (order is the contract)")



## Schema-dependent sketch (field numbers, no names)

**Why:** classic Protobuf-style wire replaces names with numbers from a shared schema.

**How:** encode the same record with field numbers 1/2/3; assert key strings are absent.

**Expect:** dense hex; opaque without the schema; teaching subset only (not production `protoc` output for all types).

**Why it matters:** density and codegen power require maintaining the contract—evolution rules live in process/IDL, not in each payload’s field names.


In [ ]:
def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def encode_record_pb_style(r: dict) -> bytes:
    # 1:user_id varint, 2:name string, 3:balance as fixed64 bits of float64 (toy)
    import struct
    out = bytearray()
    out += encode_key(1, 0) + encode_varint(int(r["user_id"]))
    name = r["name"].encode()
    out += encode_key(2, 2) + encode_varint(len(name)) + name
    out += encode_key(3, 1) + struct.pack("<d", float(r["balance"]))  # wire type 1 = 8 bytes
    return bytes(out)


pb = encode_record_pb_style(RECORD)
print("pb-style nbytes", len(pb), "hex", hex_bytes(pb))
for key in RECORD:
    assert key.encode() not in pb
print("OK: field names absent; numbers 1/2/3 carry identity via shared schema")



## Size table (this payload only)

**Why:** one table makes the spectrum concrete for *this* record—not a universal ranking.

**How:** print nbytes for each encoding you built above.

**Expect:** JSON ≥ MessagePack map > (array / pb-style), with exact order depending on values.

**Why it matters:** self-describing vs schema-dependent is a **spectrum**; pick the point that matches inspectability vs contract investment.


In [ ]:
rows = [("JSON", len(j)), ("pb-style sketch", len(pb))]
if HAS_MSGPACK:
    rows.insert(1, ("MessagePack map", len(msgpack.packb(RECORD, use_bin_type=True))))
    rows.insert(2, ("MessagePack array", len(msgpack.packb(list(RECORD.values()), use_bin_type=True))))
for name, n in rows:
    print(f"{name:20} {n:4} bytes")



## Takeaways

- **Self-describing:** meaning partly in the payload (names/tags).
- **Schema-dependent:** meaning in the shared contract; wire is denser and opaque alone.
- Spectrum: JSON → MessagePack map → MessagePack array → Protobuf field numbers.

**Why it matters:** design reviews should ask “who carries field identity?” before arguing about milliseconds.

**Next:** [Schema evolution](./schema_evolution.ipynb)
